# Da base à tabela

Como exemplo, este notebook usa uma base com 22.866 sentenças do TJSP em ações contra bancos. O cliente quer saber em quantas o pedido foi negado, e por quê. Este notebook faz o caminho inteiro numa amostra pequena: carregar a base, estimar o custo, pedir ao modelo uma resposta no formato de uma tabela e conferir se ele acertou.

O arquivo `dataset_clinica_20261.csv`, com o link no site da aula, precisa estar na mesma pasta deste notebook. No Colab, abra o painel "Arquivos" (o ícone de pasta na barra lateral) e arraste o arquivo para lá; ele some quando a sessão termina.

## 1. Instalar

O `dataframeit` aplica um modelo de linguagem a cada linha de uma tabela. O extra `groq` instala a integração com o Groq, que tem plano gratuito.

In [ ]:
%pip install "dataframeit[groq]" pandas python-dotenv

## 2. A chave do Groq

1. Crie uma conta em https://console.groq.com e gere uma chave em "API Keys".
2. Na pasta deste notebook, crie um arquivo chamado `.env` com uma linha só: `GROQ_API_KEY=sua_chave_aqui`.
3. Não compartilhe o `.env` nem cole a chave numa célula. Quem tiver a chave usa a sua conta.

No Colab não há `.env`: use o ícone de chave na barra lateral ("Secrets"), crie o segredo `GROQ_API_KEY` com a sua chave e ligue o acesso deste notebook. A célula abaixo funciona nos dois lugares.

## 3. Uma pergunta ao modelo

A chave não aparece no código: o `ChatGroq` a lê do ambiente.

## 4. A base e o custo

Cada 4 caracteres valem cerca de 1 token. A base tem textos de decisão repetidos: eles saem antes de qualquer conta, porque mandar o mesmo texto duas vezes custa duas vezes.

## 5. Uma amostra que cabe no plano gratuito

O plano gratuito aceita 8 mil tokens por minuto. Uma sentença maior que isso não passa sozinha, e por isso a amostra fica só com as de até 6 mil tokens. Com semente fixa, rodar de novo sorteia as mesmas sentenças.

## 6. A tabela que queremos de volta

Cada campo da classe vira uma coluna. `Literal` obriga o modelo a escolher uma das opções; `list[Literal]` deixa escolher nenhuma, uma ou várias; `bool` pede `True` ou `False`. A `description` de cada campo vai ao modelo e diz o que coletar: é ali que se explica o que conta como cada opção. Esta é a classe já corrigida depois da conferência da aula.

O prompt diz a tarefa; o que conta como cada opção já está nas descriptions. `{texto}` é trocado pelo texto de cada linha.

## 7. Rodar

`rate_limit_delay` espera alguns segundos entre uma sentença e outra, para não passar do limite por minuto do plano gratuito. As dez levam alguns minutos.

## 8. O modelo acertou?

Leia o dispositivo de cada sentença, que a célula abaixo mostra a partir do último "ante o exposto" ou "posto isso", e anote o resultado na lista `leitura`, na mesma ordem da tabela. Use exatamente as opções da classe. Se o trecho mostrado não trouxer o dispositivo, leia a sentença inteira em `tabela["decisao"][numero]`.

Uma concordância numa amostra estima a base inteira e não a garante: as linhas que ninguém leu continuam sem conferência. Se o erro tiver padrão, ajuste o prompt e confira de novo numa amostra nova.

## 9. Guardar

O resultado vai para um CSV, para não pagar de novo por uma resposta que já veio.